# Assignment 16: NLP Text Preprocessing Pipeline
### Author: Parth Dadhaniya

**Dataset Information:**
- **Dataset Source:** Self-curated Customer Feedback and Product Reviews dataset containing 25 realistic samples with raw text noise (URLs, email addresses, HTML tags, emojis, numbers, punctuation, casing, and stopwords).
- **Dataset File:** `customer_reviews.csv`
- **Objective:** Implement text cleaning, tokenization, stemming, lemmatization, and an end-to-end preprocessing pipeline for NLP.

### Task 1: Load Text Dataset and Identify Common Raw Text Issues

In [ ]:
import pandas as pd

# load dataset
df = pd.read_csv('customer_reviews.csv')
print('Dataset Shape:', df.shape)
print('\n--- First 5 Text Samples and Lengths ---')
for i, row in df.head(5).iterrows():
    print(f'Sample {row["id"]} [{row["category"]}] (Length: {len(row["raw_text"])} chars):')
    print(' ', row['raw_text'])

print('\nCommon Raw Text Issues Identified:')
print('- HTML tags and entities (<div>, <b>, &amp;)')
print('- Full URLs (https://shop.com/...) and email addresses (support@gadgets.com)')
print('- Non-ASCII symbols and emojis (🔥, 👍, 😡)')
print('- Punctuation clusters (!!!, ...)')
print('- Numbers and dates (24, 14, 2024, #90210)')
print('- Inconsistent casing and frequent stopwords')

### Task 2: Basic Cleaning (Stored in clean_text_basic)

In [ ]:
import re

def basic_clean(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\d+', '', text)
    return re.sub(r'\s+', ' ', text).strip()

df['clean_text_basic'] = df['raw_text'].apply(basic_clean)

print('--- Basic Cleaning Samples ---')
for i in range(3):
    print(f'\n[Raw Sample {i+1}]:', df['raw_text'].iloc[i])
    print(f'[clean_text_basic {i+1}]:', df['clean_text_basic'].iloc[i])

### Task 3: Advanced Noise Removal (Stored in clean_text_advanced)

In [ ]:
def advanced_noise_removal(text):
    # remove HTML tags & entities
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'&\w+;', '', text)
    # remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    # remove emails
    text = re.sub(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b', '', text)
    # remove emojis and non-ascii
    text = text.encode('ascii', 'ignore').decode('ascii')
    # remove punctuation and numbers
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\d+', '', text)
    return re.sub(r'\s+', ' ', text).strip().lower()

df['clean_text_advanced'] = df['raw_text'].apply(advanced_noise_removal)

print('--- Advanced Cleaning Samples ---')
for i in range(3):
    print(f'\n[clean_text_advanced {i+1}]:', df['clean_text_advanced'].iloc[i])

### Task 4: Stopword Removal (Stored in text_no_stopwords)

In [ ]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    words = text.split()
    return ' '.join([w for w in words if w not in stop_words])

df['text_no_stopwords'] = df['clean_text_advanced'].apply(remove_stopwords)

print('--- Stopword Removal Samples ---')
for i in range(3):
    before = df['clean_text_advanced'].iloc[i]
    after = df['text_no_stopwords'].iloc[i]
    print(f'\nSample {i+1} (Words: {len(before.split())} -> {len(after.split())}):')
    print(' After Stopwords:', after)

### Task 6: Word and Sentence Tokenization (3 Samples)

In [ ]:
from nltk.tokenize import sent_tokenize, word_tokenize

for sample_id in [2, 7, 10]:
    raw = df[df['id'] == sample_id]['raw_text'].iloc[0]
    sentences = sent_tokenize(raw)
    words = word_tokenize(raw)
    
    print(f'\n================== Sample {sample_id} ==================')
    print('Raw Text:', raw)
    print(f'\nSentences ({len(sentences)}):')
    for s in sentences:
        print(' -', s)
    print(f'\nWord Tokens ({len(words)}):')
    print(' ', words[:12], '...')

### Task 7: Word Stemming using Porter Stemmer

In [ ]:
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()
sample_words = ['running', 'studies', 'attentive', 'carefully', 'happily', 'leaves', 'connected', 'batteries']

print(f'{"Original":<15} | {"Stemmed":<15}')
print('-' * 32)
for w in sample_words:
    print(f'{w:<15} | {stemmer.stem(w):<15}')

### Task 8: WordNet Lemmatization vs Porter Stemming

In [ ]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

test_words = [
    ('running', 'v'),
    ('studies', 'n'),
    ('leaves', 'n'),
    ('better', 'a'),
    ('corpora', 'n'),
    ('caring', 'v'),
    ('happily', 'r'),
    ('batteries', 'n')
]

print(f'{"Word":<12} | {"POS":<5} | {"Porter Stemmer":<16} | {"WordNet Lemmatizer":<20}')
print('-' * 60)
for w, pos in test_words:
    print(f'{w:<12} | {pos:<5} | {stemmer.stem(w):<16} | {lemmatizer.lemmatize(w, pos=pos):<20}')

### Task 9: End-to-End nlp_preprocess(text) Pipeline

In [ ]:
def nlp_preprocess(text):
    if not isinstance(text, str):
        return ''
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'&\w+;', '', text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b', '', text)
    text = text.encode('ascii', 'ignore').decode('ascii')
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\d+', '', text)
    tokens = text.lower().strip().split()
    cleaned = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words and len(t) > 1]
    return ' '.join(cleaned)

df['final_clean_text'] = df['raw_text'].apply(nlp_preprocess)

print('--- Final Pipeline Results ---')
for i in range(5):
    print(f'\n[Sample {i+1} Raw]:        ', df['raw_text'].iloc[i])
    print(f'[Sample {i+1} Final Clean]:', df['final_clean_text'].iloc[i])

df.to_csv('cleaned_reviews_final.csv', index=False)
print('\nSaved final cleaned dataset to cleaned_reviews_final.csv')

### Task 10: Technical Observations & Analysis

1. **Basic vs. Advanced Cleaning:**
   - Basic cleaning strips punctuation mechanically, leaving broken URL fragments and tag residuals.
   - Advanced noise removal targets complete structural entities (URLs, emails, HTML tags, and emojis) before word-level operations, keeping the vocabulary clean.

2. **Lemmatization vs. Stemming:**
   - Stemming uses fast suffix chopping without dictionary checks, often producing non-words (`studies` $\to$ `studi`).
   - Lemmatization uses morphological analysis with POS tagging to resolve words to their canonical dictionary root (`studies` $\to$ `study`, `better` $\to$ `good`).

3. **Importance of Text Preprocessing:**
   - Drastically reduces vocabulary dimension (by over 30%).
   - Suppresses non-generalizable noise (emails, domain names).
   - Improves downstream model convergence and accuracy.